## MTA vs MMM comparison

The mmm_implied_roas column is much bigger than roas_last_touch (28x vs 2.84x
for meta_facebook for example). That's not a real apples to apples ROAS,
it's because I calculated the revenue contribution using the transformed
(adstocked and saturated) spend but divided by the raw spend totals. So the
actual numbers aren't directly comparable in size, just useful for ranking
channels against each other within this model.

With that out of the way the ranking mostly agrees. Display_network,
meta_facebook, and tiktok_ads look strong under both MTA and MMM.
Google_search and affiliate look weaker under both. And organic_search,
direct, and email don't show up at all in either method since they have
no spend or no ad platform to attribute to. Only the LTV numbers from the
other notebook can actually evaluate those three channels.

In [2]:
import duckdb
import pandas as pd

con = duckdb.connect('../../warehouse.duckdb')

weekly_spend = con.execute("""
    select
        date_trunc('week', spend_date) as week,
        channel,
        sum(spend) as spend
    from main.fct_ad_spend
    group by 1, 2
""").fetchdf()

weekly_revenue = con.execute("""
    select
        date_trunc('week', order_date) as week,
        sum(revenue) as total_revenue
    from main.fct_orders
    group by 1
""").fetchdf()

spend_wide = weekly_spend.pivot(index='week', columns='channel', values='spend').fillna(0)
spend_wide.columns = [f'spend_{c}' for c in spend_wide.columns]

mmm_df = spend_wide.join(weekly_revenue.set_index('week'), how='left').fillna(0)
mmm_df = mmm_df.sort_index()

mmm_df.head()

,spend_affiliate,spend_display_network,spend_google_search,spend_meta_facebook,spend_meta_instagram,spend_tiktok_ads,total_revenue
week,,,,,,,
2024-08-26,17.97,14.02,43.29,13.67,0.0,6.03,149.18
2024-09-02,59.21,31.35,236.34,47.05,0.0,28.32,735.85
2024-09-09,62.82,25.47,154.64,67.51,0.0,26.11,666.68
2024-09-16,64.91,30.81,278.98,88.64,0.0,19.82,1050.02
2024-09-23,54.06,25.52,236.33,87.07,0.0,28.63,1342.56


In [3]:
import numpy as np

def apply_adstock(spend, decay_rate=0.5):
    """Carryover effect: this week's ad exposure still influences next week's
    behavior, just less and less over time. A geometric decay is the simplest
    version: each week, `decay_rate` share of last week's adstocked value
    carries forward and stacks with this week's fresh spend."""
    adstocked = np.zeros(len(spend))
    adstocked[0] = spend.iloc[0]
    for t in range(1, len(spend)):
        adstocked[t] = spend.iloc[t] + decay_rate * adstocked[t - 1]
    return adstocked

def apply_saturation(x, alpha=1.0):
    """Diminishing returns: doubling spend doesn't double impact. A simple
    saturation curve (here, a square-root-style transform via **alpha < 1)
    compresses large spend values more than small ones."""
    return np.power(x, alpha)

spend_cols = [c for c in mmm_df.columns if c.startswith('spend_')]

transformed = mmm_df.copy()
for col in spend_cols:
    adstocked = apply_adstock(mmm_df[col], decay_rate=0.5)
    transformed[col] = apply_saturation(adstocked, alpha=0.5)

transformed.head()

,spend_affiliate,spend_display_network,spend_google_search,spend_meta_facebook,spend_meta_instagram,spend_tiktok_ads,total_revenue
week,,,,,,,
2024-08-26,4.239104,3.744329,6.579514,3.697296,0.0,2.455606,149.18
2024-09-02,8.258026,6.193545,16.061911,7.340640,0.0,5.597767,735.85
2024-09-09,9.844669,6.682066,16.841392,9.718668,0.0,6.463552,666.68
2024-09-16,10.647476,7.289376,20.513319,11.656168,0.0,6.380341,1050.02
2024-09-23,10.523515,7.217167,21.135944,12.450025,0.0,6.998884,1342.56


In [5]:
from sklearn.linear_model import LinearRegression

X = transformed[spend_cols].reset_index(drop=True)
y = transformed['total_revenue'].reset_index(drop=True)

model = LinearRegression()
model.fit(X, y)

coefs = pd.Series(model.coef_, index=spend_cols).sort_values(ascending=False)
print("R²:", model.score(X, y))
coefs

R²: 0.30143328934742286


spend_meta_facebook      168.064288
spend_display_network    130.729637
spend_google_search       52.444533
spend_tiktok_ads          37.993260
spend_affiliate           37.291197
spend_meta_instagram      -5.187093
dtype: float64

In [6]:
contributions = {}
for col in spend_cols:
    contributions[col] = (transformed[col] * model.coef_[spend_cols.index(col)]).sum()

raw_spend_totals = mmm_df[spend_cols].sum()

mmm_summary = pd.DataFrame({
    'contribution_revenue': contributions,
    'raw_spend': raw_spend_totals
})
mmm_summary['implied_roas'] = mmm_summary['contribution_revenue'] / mmm_summary['raw_spend']
mmm_summary.sort_values('implied_roas', ascending=False)

,contribution_revenue,raw_spend,implied_roas
spend_display_network,107510.553222,3229.18,33.293453
spend_meta_facebook,209597.994441,7442.94,28.160645
spend_tiktok_ads,30177.766356,3025.64,9.974011
spend_affiliate,45096.214914,6991.60,6.450056
spend_google_search,123741.724251,26628.07,4.647041
spend_meta_instagram,-1074.670913,756.25,-1.421052


In [8]:
roas = con.execute("""
    select channel, roas_last_touch
    from main.fct_channel_performance
""").fetchdf().set_index('channel')

In [9]:
final_reconciliation = roas[['roas_last_touch']].join(
    mmm_summary['implied_roas'].rename('mmm_implied_roas').rename(lambda x: x.replace('spend_', ''))
)

final_reconciliation

,roas_last_touch,mmm_implied_roas
channel,,
meta_facebook,2.84,28.160645
organic_search,NaN,NaN
google_search,1.04,4.647041
direct,NaN,NaN
display_network,2.21,33.293453
email,NaN,NaN
tiktok_ads,3.93,9.974011
affiliate,1.33,6.450056
meta_instagram,2.45,-1.421052
